**Use appropriate algorithm and perform sentiment analysis for Indian Language.**

In [7]:
!pip install transformers[sentencepiece] sacremoses -q

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

# 1. Sentiment (This pipeline usually works because 'sentiment-analysis' is a standard task)
sentiment_task = pipeline("sentiment-analysis", model="lxyuan/distilbert-base-multilingual-cased-sentiments-student")

# 2. Manual Setup for Translation (No pipeline used here)
model_name = "Helsinki-NLP/opus-mt-hi-en"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("\n[SUCCESS] Models loaded manually. We will bypass the 'pipeline' task registry.")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning



[SUCCESS] Models loaded manually. We will bypass the 'pipeline' task registry.


In [8]:
def translate_manual(text_list):
    # Tokenize the input
    inputs = tokenizer(text_list, return_tensors="pt", padding=True, truncation=True)

    # Generate the translation
    with torch.no_grad():
        generated_tokens = model.generate(**inputs)

    # Decode the output back to strings
    return tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)

def analyze_indic_text(text_list):
    # Get Sentiments
    sentiments = sentiment_task(text_list)

    # Get Translations manually
    translations = translate_manual(text_list)

    report = []
    for i in range(len(text_list)):
        report.append({
            "original": text_list[i],
            "translation": translations[i],
            "sentiment": sentiments[i]['label'],
            "score": round(sentiments[i]['score'], 4)
        })
    return report

In [9]:
# Test sentences in Hindi
hindi_samples = [
    "यह बहुत अच्छा है!",
    "मुझे यह बिल्कुल पसंद नहीं आया।"
]

results = analyze_indic_text(hindi_samples)

print(f"{'HINDI':<30} | {'ENGLISH':<30} | {'SENTIMENT'}")
print("-" * 80)
for res in results:
    print(f"{res['original']:<30} | {res['translation']:<30} | {res['sentiment']} ({res['score']})")

HINDI                          | ENGLISH                        | SENTIMENT
--------------------------------------------------------------------------------
यह बहुत अच्छा है!              | That's great!                  | positive (0.8428)
मुझे यह बिल्कुल पसंद नहीं आया। | I don't like it at all.        | positive (0.4292)
